<a href="https://colab.research.google.com/github/cras-lab/LangChain/blob/main/Research_AI_AGENT_FromWEB.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

필요한 모델을 설치한다.

In [ ]:
!pip install -q langchain langchain-openai langchain-community ddgs

DuckDuckGoSearchAPIWrapper<BR>
API 키 필요 없음 (대부분 경우) 가볍게 검색 기능 붙이기 좋아 개발자들이 많이이용<BR>
구글: 검색 품질과 커버리지가 강하고, 개인화·광고 생태계가 강함<BR>
DuckDuckGo: 프라이버시가 강하고, 비개인화·비추적 검색을 선호할 때 유리

In [ ]:
from langchain_community.utilities import DuckDuckGoSearchAPIWrapper
from typing import List

API 처리와 웹 문서 처리를 위한 모듈 설치

In [ ]:
import requests
from bs4 import BeautifulSoup

쿼리에 해당하는 결과 link 주소를 num_results개 반환하는 함수를 정의

In [ ]:
def web_search(
  web_query: str,
  num_results: int) -> List[str]:
  return [r["link"] for r in DuckDuckGoSearchAPIWrapper().results(web_query, num_results)]

URL을 주면, 실제 해당 웹사이트에서 scrawl 해 오는 함수 정의<BR>
=======================================================<BR>
requests.get(url)만 보내면 파이썬 requests  요청이라는 것이 드러남<BR>

서버 입장에서는 봇/스크립트 요청으로 판단 차단하거나 축약된 페이지 반환 또는<BR>아예 다른 응답 반환 할 수 있음<BR>

headers를 넣어 브라우저에서 접속한 것처럼 보이게 한다.

In [ ]:
def web_scrape(url: str) -> str:
  try:
    headers = {
      "User-Agent": (
      "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
      "AppleWebKit/537.36 (KHTML, like Gecko) "
      "Chrome/124.0.0.0 Safari/537.36"
      ),
      "Accept-Language": "en-US,en;q=0.9",
    }
    response = requests.get(url, headers=headers, timeout=15)

    if response.status_code == 200:
      soup = BeautifulSoup(response.text, "html.parser")
      page_text = soup.get_text(separator=" ", strip=True)
      return page_text
    else:
      return f"Could not retrieve the webpage: Status code {response.status_code}"

  except Exception as e:
    print(e)
    return f"Could not retrieve the webpage: {e}"

필요한 LangChain 모듈 임포트

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_community.utilities import WikipediaAPIWrapper
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

외부에서 Open API 키를 입력 받는다.

In [3]:
import getpass
OPENAI_API_KEY = getpass.getpass("YOUR KEY: ")
llm = ChatOpenAI(openai_api_key=OPENAI_API_KEY, model_name="gpt-5-nano", temperature=0)

JSON 문자열 s를 파이썬 객체로 바꾸고, 실패하면 빈 딕셔너리를 반환하는 함수정의

In [41]:
import json
def to_obj(s):
  try:
    return json.loads(s)
  except Exception:
    return {}

다양한 역할에 대한 지침 부여

In [42]:
ASSISTANT_SELECTION_INSTRUCTIONS = """
너는 연구 질문을 적절한 연구 보조원에게 할당하는 데 능숙하다.
여러 명의 연구 보조원이 존재하며, 각 보조원은 특정 전문 분야를 가지고 있다.

각 보조원은 고유한 유형으로 식별된다.
또한 각 보조원은 연구를 수행하기 위한 구체적인 지침을 가지고 있다.

올바른 보조원을 선택하는 방법은 다음과 같다.
질문의 주제에 따라 적절한 보조원을 선택해야 하며,
이때 질문의 주제는 보조원의 전문 분야와 일치해야 한다.
Examples:
질문: "Apple 주식에 투자해야 할까?"
응답:
{{
"assistant_type": "금융 분석가 보조원",
"assistant_instructions": "당신은 숙련된 금융 분석가 AI 보조원이다. 주요 목표는
제공된 데이터와 추세를 바탕으로 포괄적이고 통찰력 있으며, 편향되지 않고 체계적으로 구성된 금융 보고서를 작성하는 것이다.",
"user_question": {user_question}
}}

질문: "텔아비브에서 가장 흥미로운 관광지는 어디인가?"
응답:
{{
"assistant_type": "여행 가이드 보조원",
"assistant_instructions": "당신은 세계를 여행한 AI 여행 가이드 보조원이다. 주요 목적은 특정 지역에 대해 역사, 명소,
문화적 통찰을 포함하여 흥미롭고 통찰력 있으며,
편향되지 않고 구조적으로 잘 정리된 여행 보고서를 작성하는 것이다.",
"user_question": "{user_question}"
}}
}}


질문: "{user_question}"
응답:
{{
"assistant_type": "기술 분석 보조원",
"assistant_instructions": "당신은 AI, 프로그래밍, 데이터 과학 및 엔지니어링 분야를 전문으로 하는 기술 분석 AI 보조원이다.
주요 목적은 기술적 개념, 모델, 시스템 및 코드에 대해 정확하고 체계적이며 깊이 있는 분석 보고서를 작성하는 것이다. 설명은 논리적 구조를 갖추고,
필요 시 수식, 알고리즘, 아키텍처 및 실제 구현 관점을 포함해야 한다.",
"user_question": "{user_question}"
}}

질문: "메시는 좋은 축구 선수인가?"
응답:
{{
"assistant_type": "스포츠 전문가 보조자",
"assistant_instructions": "당신은 경험 많은 AI 스포츠 보조자다. 주요 목적은 주어진 스포츠 인물이나 스포츠 이벤트에 대해 사실적 세부사항,
통계, 통찰을 포함하여 흥미롭고 통찰력 있으며 편향되지 않고 구조화된 스포츠 보고서를 작성하는 것이다.",
"user_question": "{user_question}"
}}
--

이제 위의 모든 내용을 이해했으므로, 다음 질문에 대해 올바른 연구 보조자를 선택하라.
질문: {user_question}
응답:

"""

비서유형 선택을 위한 프롬프트 생성기를 호출한다.<BR>
다음과 같은 형식으로 사용자의 프롬프트를 구조화 한다(변수목록 자동추출).<BR>
변수를 파싱 metadata로 정리만 하고 문장을 해석하거나 요약하지는 않는다.<BR>
======================================= <BR>
input_variables=['user_question'] <BR>
input_types={} <BR>
partial_variables={} <BR>
template='...' <BR>

In [43]:
ASSISTANT_SELECTION_PROMPT_TEMPLATE = PromptTemplate.from_template(
  template=ASSISTANT_SELECTION_INSTRUCTIONS
)

In [ ]:
print(ASSISTANT_SELECTION_PROMPT_TEMPLATE)

웹 서치를 위한 지침 정의

In [ ]:
WEB_SEARCH_INSTRUCTIONS = """
{assistant_instructions}

   질문:{user_question}에 대해 가능한 한 많은 정보를 수집하기 위해
  {num_search_queries}개의 웹 검색 쿼리를 작성하라.

  목표는 검색을 통해 얻은 정보를 기반으로 보고서를 작성하는 것이다.

반드시 아래 형식으로 query1, query2, query3과 같은
쿼리 목록 형태로 응답해야 한다:

[
  {"search_query": "query1", "user_question": "{user_question}" },
  {"search_query": "query2", "user_question": "{user_question}" },
  {"search_query": "query3", "user_question": "{user_question}" }
]
"""

웹 서치를 위한 프롬프트 생성기 호출



In [ ]:
WEB_SEARCH_PROMPT_TEMPLATE = PromptTemplate.from_template(
  template=WEB_SEARCH_INSTRUCTIONS
)

요약을 위한 지침을 설정

In [ ]:
SUMMARY_INSTRUCTIONS = """
  다음 텍스트를 읽어라:
  텍스트: {search_result_text}
-----------
위 텍스트를 바탕으로 다음 질문에 대해 간단히 답하라.
질문: {search_query}
만약 위 질문에 대해 제공된 텍스트만으로 답할 수 없다면,
텍스트를 요약하라.
가능한 경우 모든 사실 정보, 수치, 통계 등을 포함하라.
"""

요약을 위한 프롬프트 생성기 호출

In [7]:
SUMMARY_PROMPT_TEMPLATE = PromptTemplate.from_template(
  template=SUMMARY_INSTRUCTIONS
)

최종 리포트 생성을 위한 지침 정의

In [8]:
RESEARCH_REPORT_INSTRUCTIONS = """
너는 비판적 사고를 수행하는 AI 연구 보조원이다.
너의 유일한 목적은 주어진 텍스트를 기반으로
잘 작성되고, 높은 평가를 받을 수 있으며,
객관적이고 구조적인 보고서를 작성하는 것이다.

정보:
--------
{research_summary}
--------

위 정보를 바탕으로 다음 질문 또는 주제에 대해
"{user_question}"에 대한 상세한 보고서를 작성하라.

보고서는 다음 기준을 충족해야 한다:
- 질문에 대한 답변에 집중할 것
- 구조가 명확하고 정보성이 높을 것
- 충분히 깊이 있는 내용을 포함할 것
- 가능한 경우 사실, 수치, 통계를 포함할 것
- 최소 1,200단어 이상으로 작성할 것

제공된 모든 관련 정보와 필요한 정보를 최대한 활용하여
가능한 한 길고 충실하게 작성해야 한다.

보고서는 반드시 Markdown 문법으로 작성해야 한다.

주어진 정보를 기반으로 구체적이고 타당한 자신의 의견을 반드시 도출해야 한다.
일반적이고 의미 없는 결론을 추론해서는 안 된다.

사용된 모든 출처 URL은 보고서 마지막에 작성하되,
중복 없이 하나씩만 포함해야 한다.

보고서는 APA 형식으로 작성해야 한다.

최선을 다해 작성하라. 이는 매우 중요한 작업이다.
"""

리포트 생성을 위한 템플릿 생성기 호출

In [ ]:
RESEARCH_REPORT_PROMPT_TEMPLATE = PromptTemplate.from_template(
    template=RESEARCH_REPORT_INSTRUCTIONS
)

웹 검색을 위한 지침 정의

In [9]:
WEB_SEARCH_INSTRUCTIONS = """
{assistant_instructions}
다음 질문에 대해 가능한 한 많은 정보를 수집하기 위해
{num_search_queries}개의 웹 검색 질의를 작성하라:
{user_question}. 목표는 수집한 정보를 바탕으로 보고서를 작성하는 것이다.
다음 형식으로 query1, query2, query3과 같은 질의 목록으로 응답해야 한다:
[
{{"search_query": "query1", "user_question": "{user_question}" }},
{{"search_query": "query2", "user_question": "{user_question}" }},
{{"search_query": "query3", "user_question": "{user_question}" }}
]
"""


웹 검색을 위한 템플릿 생성



In [ ]:
WEB_SEARCH_PROMPT_TEMPLATE = PromptTemplate.from_template(
  template=WEB_SEARCH_INSTRUCTIONS
)

연구리포트 주제를 정의한다.

In [45]:
question = '최근 일주일 동안의 KOSPI 투자자 동향 찾아서 주체별로 정리해 봐'

연구주제에 맞는 프롬프트를 찾고 해당 지침을 작성

In [46]:
assistant_selection_prompt = ASSISTANT_SELECTION_PROMPT_TEMPLATE.format(user_question=question)
assistant_instructions = llm.invoke(assistant_selection_prompt)

In [47]:
NUM_SEARCH_QUERIES = 2
NUM_SEARCH_RESULTS_PER_QUERY = 3
RESULT_TEXT_MAX_CHARACTERS = 10000

In [ ]:
print(assistant_instructions)

In [49]:
assistant_instructions_dict = to_obj(assistant_instructions.content)

In [ ]:
print(assistant_instructions_dict)

In [51]:
web_search_prompt = WEB_SEARCH_PROMPT_TEMPLATE.format(
  assistant_instructions=assistant_instructions_dict['assistant_instructions'],
  num_search_queries=NUM_SEARCH_QUERIES,
  user_question=assistant_instructions_dict['user_question'])

web_search_queries = llm.invoke(web_search_prompt)
web_search_queries_list = to_obj(web_search_queries.content.replace('\n', ''))

In [ ]:
print( web_search_queries_list )

In [53]:
searches_and_result_urls = [{
  'result_urls': web_search(web_query=wq['search_query'],
          num_results=NUM_SEARCH_RESULTS_PER_QUERY),
          'search_query': wq['search_query']}

  for wq in web_search_queries_list]

In [ ]:
print( searches_and_result_urls )

In [55]:
search_query_and_result_url_list = []
for qr in searches_and_result_urls:
  search_query_and_result_url_list.extend([{
    'search_query': qr['search_query'],
    'result_url': r}
      for r in qr['result_urls']])

In [ ]:
print( search_query_and_result_url_list )

In [57]:
result_text_list = [{
  'result_text': web_scrape(
  url=re['result_url'])[:RESULT_TEXT_MAX_CHARACTERS],
  'result_url': re['result_url'],
  'search_query': re['search_query']}
  for re in search_query_and_result_url_list]

In [ ]:
print( result_text_list )

In [59]:
result_text_summary_list = []

for rt in result_text_list:
  summary_prompt = SUMMARY_PROMPT_TEMPLATE.format(
  search_result_text=rt['result_text'],
  search_query=rt['search_query'])
  text_summary = llm.invoke(summary_prompt)

  result_text_summary_list.append({
    'text_summary': text_summary,
    'result_url': rt['result_url'],
    'search_query': rt['search_query']})

In [ ]:
print( result_text_summary_list )

In [61]:
RESEARCH_REPORT_INSTRUCTIONS = """
너는 비판적 사고를 수행하는 AI 연구 보조자다.
유일한 목적은 주어진 텍스트에 대해 잘 작성되고,
비평적으로 우수하며, 객관적이고 구조화된
보고서를 작성하는 것이다.
정보:
---
## {research_summary}

위 정보를 활용하여 다음에 답하라:
…
…
"""

In [62]:
stringified_summary_list = [
  f'Source URL: {sr["result_url"]}\nSummary: {sr["text_summary"]}'
    for sr in result_text_summary_list]

In [ ]:
for i, item in enumerate(result_text_summary_list, 1):
    print(f"\n--- {i} ---")
    print(item["text_summary"].content)

Next, combine all the summary strings into one:

In [64]:
appended_result_summaries = '\n'.join(stringified_summary_list)

In [ ]:
print( appended_result_summaries )

In [66]:
research_report_prompt = RESEARCH_REPORT_PROMPT_TEMPLATE.format(
  research_summary=appended_result_summaries,
  user_question=question
)

research_report = llm.invoke(research_report_prompt)



In [ ]:
print(research_report.content)